# PR-CLIP v3 FIXED — End-to-end dual residual adapters (1 epoch)

**Đã khôi phục train loop gốc:** `training.main` → model forward → InfoNCE + reconstruction → AMP accumulation → RSICD retrieval evaluation → checkpoint.

Không dùng feature cache, không thay bằng external script train adapter. Cấu hình: RET-2, batch 16, one epoch, backbone LR `2e-7`, adapter LR `2e-5` (100×). Reconstruction gốc giữ nguyên. Adapter được áp dụng trong `encode_image` và `encode_text`, do đó dùng cả khi evaluation.

**Lưu ý:** Đây là một phép thử kiến trúc, chưa có kết quả thực nghiệm. Kết quả nên được chọn theo validation độc lập; repo hiện tại có rsicd_val.csv trùng test.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

# upload PR-CLIP.zip lên Drive trước, sửa đường dẫn cho đúng
!rm -rf /content/PR-CLIP
!unzip -q -o /content/drive/MyDrive/PR-CLIP.zip -d /content
%cd /content/PR-CLIP

Mounted at /content/drive
/content/PR-CLIP


In [2]:
!pip install -q open_clip_torch==2.20.0 clip-benchmark==1.6.1 transformers==4.45.2 adapters timm sentence-transformers wandb pyyaml gdown
!pip install -q "protobuf>=6.31.1,<7"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-api-core 2.30.3 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-mon

In [3]:
!gdown --fuzzy "https://drive.google.com/file/d/15htMe2iE4kVujjWURxfCxZXk8vAAetoT/view" -O images.zip
!rm -rf images && mkdir images
!tar -xzf images.zip -C images
!find images -type f -exec mv -t images {} + 2>/dev/null
!find images -mindepth 1 -type d -empty -delete
!rm images.zip
!mkdir -p cache/weights

Downloading...
From (original): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT
From (redirected): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT&confirm=t&uuid=8b61dadf-e349-4a0c-8053-693c73c5595a
To: /content/PR-CLIP/images.zip
100% 1.46G/1.46G [00:32<00:00, 45.1MB/s]


In [4]:
!mkdir -p /content/PR-CLIP/cache/weights
!wget -O /content/PR-CLIP/cache/weights/RET-2.pt \
  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
!ls -la /content/PR-CLIP/cache/weights/RET-2.pt   # khoảng 605MB

--2026-11-03 15:25:10--  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
Resolving huggingface.co (huggingface.co)... 99.86.18.96, 99.86.18.52, 99.86.18.89, ...
Connecting to huggingface.co (huggingface.co)|99.86.18.96|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/655fb2aadc7511e22748819d/48ec26e685f2b7622075158c0649039d64cdcbdbe150faedb6e79ea1d13d0e78?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27RS5M_ViT-B-32_RET-2.pt%3B+filename%3D%22RS5M_ViT-B-32_RET-2.pt%22%3B&xip=gebrgTs3HmA&X-Xet-Cas-Uid=public&Expires=1793723110&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU1ZmIyYWFkYzc1MTFlMjI3NDg4MTlkLzQ4ZWMyNmU2ODVmMmI3NjIyMDc1MTU4YzA2NDkwMzlkNjRjZGNiZGJlMTUwZmFlZGI2ZTc5ZWExZDEzZDBlNzhcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomeGlwPWdlYnJnVHMzSG1BJlgtWGV0LUNhcy1VaWQ9cHVibGljIiwiQ29uZGl0

In [5]:
import pandas as pd, os
for f in ['dataset/train/ret2.csv','dataset/test/rsicd_test.csv','dataset/test/rsitmd_test.csv']:
    names = pd.read_csv(f)['filename'].unique()
    missing = [n for n in names if not os.path.exists(os.path.join('images', n))]
    print(f, '| tổng:', len(names), '| thiếu:', len(missing))

dataset/train/ret2.csv | tổng: 13092 | thiếu: 0
dataset/test/rsicd_test.csv | tổng: 1093 | thiếu: 0
dataset/test/rsitmd_test.csv | tổng: 452 | thiếu: 0


In [6]:
%cd /content/PR-CLIP
!sed -i 's/from timm.optim.nadam import Nadam/from timm.optim.nadam import NAdamLegacy as Nadam/' training/optimization.py
!sed -i 's/from timm.optim.radam import RAdam$/from timm.optim.radam import RAdamLegacy as RAdam/' training/optimization.py
!sed -n 5,12p training/optimization.py
!nvidia-smi | head -12
!sed -i 's/np\.string_/np.bytes_/g' data/train_data.py utils/tsne_checkpoints.py
!sed -i 's/astype(np\.float)/astype(float)/g' evaluation/linear_eval.py
!grep -n "np.bytes_" data/train_data.py

/content/PR-CLIP
from timm.optim.adafactor import Adafactor
from timm.optim.adahessian import Adahessian
from timm.optim.adamp import AdamP
from timm.optim.lookahead import Lookahead
from timm.optim.nadam import NAdamLegacy as Nadam
from timm.optim.radam import RAdamLegacy as RAdam
from timm.optim.rmsprop_tf import RMSpropTF
from timm.optim.sgdp import SGDP
Tue Nov  3 15:25:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+=============

In [7]:
!pip install -q cn_clip --no-deps
!nvidia-smi
!pip install -q faiss-cpu click

Tue Nov  3 15:25:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
%cd /content/PR-CLIP
import re

def read(p): return open(p, encoding='utf-8').read().replace('\r\n', '\n')
def write(p, s): open(p, 'w', encoding='utf-8').write(s)
def sub(path, pairs):
    s = read(path)
    for old, new in pairs:
        s = s.replace(old, new)
    write(path, s)


# 2. timm / numpy bản mới đổi tên + tắt log dài
sub('training/optimization.py', [
    ('from timm.optim.nadam import Nadam\n', 'from timm.optim.nadam import NAdamLegacy as Nadam\n'),
    ('from timm.optim.radam import RAdam\n', 'from timm.optim.radam import RAdamLegacy as RAdam\n'),
    ('logging.info("Param groups = %s" % json.dumps(parameter_group_names, indent=2))', 'pass'),
])
sub('data/train_data.py', [('np.string_', 'np.bytes_')])
sub('utils/tsne_checkpoints.py', [('np.string_', 'np.bytes_')])
sub('evaluation/linear_eval.py', [('astype(np.float)', 'astype(float)')])
sub('training/main.py', [
    ("logging.info(f'\\t{n}\\t{list(p.size())}')", 'pass'),
    ("logging.info('Model structure\\n' +str(model))", 'pass'),
    ('logging.info(f"  {name}: {val}")', 'pass'),
])

# 3. Thay return_hidden của open_clip bản sửa
helper = '''    def _vit_hidden(self, images):
        v = self.image_backbone
        x = v.conv1(images.type(v.conv1.weight.dtype))
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
        x = torch.cat([cls, x], dim=1)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)
        x = x.permute(1, 0, 2)
        x = v.transformer(x)
        x = x.permute(1, 0, 2)
        return v.ln_post(x)

'''
s = read('model/model.py')
if '_vit_hidden' not in s:
    s = s.replace("self.image_backbone(images, return_hidden=True)", "self._vit_hidden(images)")
    marker = "    def forward(self, images, texts, text_only=False):"
    s = s.replace(marker, helper + marker)
    write('model/model.py', s)

# 4. In log theo % epoch (10 -> 10%,20%,...; 50 -> 50%,100%)
LOG_PCT = 10
s = read('training/train.py')
s = s.replace("(i % 10 == 0 or batch_count == num_batches_per_epoch)",
              "(batch_count in {round(num_batches_per_epoch * k * LOG_PCT / 100) for k in range(1, 100 // LOG_PCT + 1)})")
if 'LOG_PCT = ' not in s:
    anchor = "import torch.nn.functional as F\n"
    s = s.replace(anchor, anchor + f"\nLOG_PCT = {LOG_PCT}\n", 1)
else:
    s = re.sub(r"LOG_PCT = \d+", f"LOG_PCT = {LOG_PCT}", s, count=1)
write('training/train.py', s)


# Kiểm tra: phải ra 2, 1, 2, 2, True
!grep -c "_vit_hidden" model/model.py
!grep -c "NAdamLegacy" training/optimization.py
!grep -c "np.bytes_" data/train_data.py
!grep -c "LOG_PCT" training/train.py
print('[DBG-fp32]' in read('training/train.py'))


%cd /content/PR-CLIP
import re, py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

if '[LITE-CKPT]' in s:
    print('Đã vá từ trước, không làm gì thêm.')
else:
    def sub_regex(pattern, repl, text, name):
        n = len(re.findall(pattern, text))
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return re.sub(pattern, repl, text, count=1)

    def sub_literal(old, new, text, name):
        n = text.count(old)
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return text.replace(old, new)

    # 1. Không lưu epoch_N.pt (chỉ còn best.pt + epoch_latest.pt)
    s = sub_literal('if completed_epoch == args.epochs or (', 'if False or (', s, 'epoch_N')

    # 2. epoch_latest.pt: ĐẦY ĐỦ (fp32 + optimizer + scaler), ghi qua file .tmp rồi đổi tên
    pat = r'( +)if args\.save_most_recent:\n\s+torch\.save\(\n\s+checkpoint_dict,\n\s+os\.path\.join\(args\.checkpoint_path, f"epoch_latest\.pt"\),\n\s+\)[ ]*\n'
    def rep_latest(m):
        i = m.group(1)
        return (f'{i}if args.save_most_recent:  # [LITE-CKPT]\n'
                f'{i}    _tmp = os.path.join(args.checkpoint_path, "epoch_latest.tmp")\n'
                f'{i}    torch.save(checkpoint_dict, _tmp)\n'
                f'{i}    os.replace(_tmp, os.path.join(args.checkpoint_path, "epoch_latest.pt"))\n')
    s = sub_regex(pat, rep_latest, s, 'epoch_latest')

    # 3. best.pt: NHẸ (fp16, không optimizer)
    pat = r'torch\.save\(\n(\s+)checkpoint_dict,\n(\s+)os\.path\.join\(args\.checkpoint_path, f"best\.pt"\),'
    def rep_best(m):
        return ('torch.save(\n' + m.group(1) +
                '{**checkpoint_dict, "optimizer": None, "state_dict": {k: (v.half() if v.is_floating_point() else v) for k, v in checkpoint_dict["state_dict"].items()}},\n' +
                m.group(2) + 'os.path.join(args.checkpoint_path, f"best.pt"),')
    s = sub_regex(pat, rep_best, s, 'best')

    # 4. Resume được cả khi checkpoint không có optimizer (vd best.pt nhẹ)
    s = sub_literal('                    if optimizer is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    '                    if optimizer is not None and checkpoint.get("optimizer") is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    s, 'resume')

    # 5. Train xong đủ epoch -> tự xóa epoch_latest (giữ best.pt)
    old = '    if args.wandb and is_master(args):\n        wandb.finish()\n'
    new = ('    if is_master(args) and args.save_logs:  # [AUTO-DEL]\n'
           '        for _f in ("epoch_latest.pt", "epoch_latest.tmp"):\n'
           '            _p = os.path.join(args.checkpoint_path, _f)\n'
           '            if os.path.exists(_p):\n'
           '                os.remove(_p)\n'
           '                logging.info(f"Train xong, da xoa {_p}")\n') + old
    s = sub_literal(old, new, s, 'auto_delete')

    open(p, 'w', encoding='utf-8').write(s)
    py_compile.compile(p, doraise=True)   # báo lỗi cú pháp ngay nếu vá sai
    print('Đã vá xong:', '[LITE-CKPT]' in s, '[AUTO-DEL]' in s)
    print('best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong')

# ---- Vá thêm: resume trên PyTorch >= 2.6 + dừng ngay nếu thiếu checkpoint ----
import py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

def _sub_once(old, new, text, name):
    n = text.count(old)
    assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
    return text.replace(old, new)

if 'weights_only=False' not in s:
    s = _sub_once('torch.load(args.resume, map_location=device)',
                  'torch.load(args.resume, map_location=device, weights_only=False)',
                  s, 'weights_only')
if 'Khong tim thay checkpoint' not in s:
    s = _sub_once('            logging.info("=> no checkpoint found at \'{}\'".format(args.resume))',
                  '            raise FileNotFoundError(f"Khong tim thay checkpoint: {args.resume}")',
                  s, 'resume_missing')
open(p, 'w', encoding='utf-8').write(s)
py_compile.compile(p, doraise=True)
print('Vá resume OK:', 'weights_only=False' in s, 'Khong tim thay checkpoint' in s)

print([line.strip() for line in open("training/params.py") if "--load-clip-weights" in line])



/content/PR-CLIP
2
1
2
2
False
/content/PR-CLIP
Đã vá xong: True True
best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong
Vá resume OK: True True
['"--load-clip-weights", default=True, action="store_true",', '"--load-clip-weights-from", default=\'./cache/weights/RET-2.pt\', type=str,']


## 1. Vá module + train loop (bắt buộc chạy trước train)

In [9]:
%cd /content/PR-CLIP
from pathlib import Path
import py_compile

p=Path('model/model.py'); s=p.read_text(encoding='utf-8')
cls='''class DualResidualAdapter(nn.Module):
    """Identity-initialized bottleneck adapter, acts on retrieval embeddings."""
    def __init__(self, dim=512, bottleneck=128, scale=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, bottleneck), nn.GELU(), nn.Linear(bottleneck, dim))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.scale = scale

    def forward(self, x):
        residual = self.net(x.float())
        return F.normalize(x.float() + self.scale * residual, dim=-1)

'''
needle='class ItraModel(nn.Module):'
assert s.count(needle)==1, 'Không tìm được class ItraModel gốc'
s=s.replace(needle,cls+needle,1)
needle='        self.image_backbone = image_backbone\n'
assert s.count(needle)==1
s=s.replace(needle,needle+'        self.image_adapter_v3 = DualResidualAdapter(dim=512)\n        self.text_adapter_v3 = DualResidualAdapter(dim=512)\n',1)
# Forward encoders must use adapter in training AND evaluation (same codepath).
needle='        return image_features.float()'
assert s.count(needle)==1
s=s.replace(needle,'        return self.image_adapter_v3(image_features.float())',1)
needle='        return text_features\n'
assert s.count(needle)==1
s=s.replace(needle,'        return self.text_adapter_v3(text_features)',1)
# Replace return_hidden=True broken for some open_clip versions.
helper='''    def _vit_hidden(self, images):
        v = self.image_backbone
        x = v.conv1(images.type(v.conv1.weight.dtype))
        x = x.reshape(x.size(0), x.size(1), -1).permute(0, 2, 1)
        cls = v.class_embedding.to(x.dtype) + torch.zeros(x.size(0), 1, x.size(-1), dtype=x.dtype, device=x.device)
        x = torch.cat([cls, x], dim=1)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)
        x = v.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        return v.ln_post(x)

'''
if '_vit_hidden' not in s:
    s=s.replace('    def forward(self, images, texts, text_only=False):',helper+'    def forward(self, images, texts, text_only=False):',1)
s=s.replace('self.image_backbone(images, return_hidden=True)', 'self._vit_hidden(images)')
# Correct mask for actual OpenCLIP text tokens. Keep hidden-state reconstruction.
needle='                text_features = open_clip_forward(texts)'
assert s.count(needle)==1
s=s.replace(needle,needle+'\n                self._v3_text_mask = texts.ne(0).long()',1)
needle='        text_masks = torch.ones(text_embed.size()[:-1],dtype=torch.long).to(self.device)'
assert s.count(needle)==1
s=s.replace(needle,'        text_masks = self._v3_text_mask.to(text_embed.device)\n        assert text_masks.shape == text_embed.shape[:2], (text_masks.shape, text_embed.shape)',1)
p.write_text(s,encoding='utf-8')

# Differential LR implemented through existing scheduler's lr_scale.
p=Path('training/optimization.py');s=p.read_text(encoding='utf-8')
needle='        if group_name not in parameter_group_names:'
assert s.count(needle)==1
s=s.replace(needle,'        if name.startswith(("image_adapter_v3.", "text_adapter_v3.")):\n            group_name = "v3_adapter_" + group_name\n'+needle,1)
needle='            parameter_group_names[group_name] = {'
assert s.count(needle)==1
s=s.replace(needle,'            if group_name.startswith("v3_adapter_"):\n                scale *= 100.0\n'+needle,1)
p.write_text(s,encoding='utf-8')

# AMP accumulation: optimizer step only after 4 microbatches.
p=Path('training/train.py');s=p.read_text(encoding='utf-8')
needle='    step_lou = 4\n'
assert s.count(needle)==1
s=s.replace(needle,needle+'    optimizer.zero_grad(set_to_none=True)  # V3-AMP-FIX\n    norm = torch.tensor(0.0)\n',1)
start='            if scaler is not None:\n                scaler.scale(total_loss).backward()'
end='            batch_time_m.update(time.time() - end)'
assert s.count(start)==1 and s.count(end)==1
a=s.index(start);b=s.index(end,a)
replacement='''            if scaler is not None:
                scaler.scale(total_loss).backward()
            else:
                total_loss.backward()
            _step_now = ((i + 1) % step_lou == 0 or i == train_loader_size - 1)
            if _step_now:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.max_grad_norm)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

'''
s=s[:a]+replacement+s[b:]
s=s.replace('f"LR: {optimizer.param_groups[0][\'lr\']:3f} "','f"LR: {optimizer.param_groups[0][\'lr\']:.3e} "')
p.write_text(s,encoding='utf-8')

for name in ['model/model.py','training/train.py','training/optimization.py','training/main.py']:
    py_compile.compile(name,doraise=True)
assert 'image_adapter_v3(image_features.float())' in Path('model/model.py').read_text()
assert 'text_adapter_v3(text_features)' in Path('model/model.py').read_text()
assert 'scale *= 100.0' in Path('training/optimization.py').read_text()
assert 'V3-AMP-FIX' in Path('training/train.py').read_text()
print('PASS: adapter applied in both encoders; optimizer group LR scales; AMP fix; four Python files compile')


/content/PR-CLIP
PASS: adapter applied in both encoders; optimizer group LR scales; AMP fix; four Python files compile


In [10]:
%cd /content/PR-CLIP
# Small but meaningful sanity test for identity init and learning signal.
import torch
from model.model import DualResidualAdapter
ad=DualResidualAdapter().cuda()
x=torch.randn(4,512,device='cuda',requires_grad=True)
y=ad(x)
assert torch.allclose(y, torch.nn.functional.normalize(x,dim=-1), atol=1e-6)
(y * torch.randn_like(y)).sum().backward()
assert ad.net[-1].weight.grad is not None and torch.isfinite(ad.net[-1].weight.grad).all()
print('PASS identity forward / adapter gradient')

# Ensure original full training pipeline is present.
from pathlib import Path
for f in ['training/main.py','training/train.py','evaluation/retrieval.py','model/model.py','dataset/train/ret2.csv','dataset/test/rsicd_test.csv','cache/weights/RET-2.pt']:
    assert Path(f).is_file(), f'Missing: {f}'
assert Path('cache/weights/RET-2.pt').stat().st_size > 100_000_000
print('PASS train/data/evaluation/checkpoint preflight')

/content/PR-CLIP


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


PASS identity forward / adapter gradient
PASS train/data/evaluation/checkpoint preflight


## 2. Huấn luyện END-TO-END (torchrun) — có evaluation và lưu checkpoint

In [11]:
%cd /content/PR-CLIP
import subprocess, sys
NAME='rsicd_v3_FIXED_AdapterEndToEnd_ep1_bs16_lr2e7'
cmd=[sys.executable,'-m','training.main',
 '--train-data','./dataset/train/ret2.csv','--images-dir','./images',
 '--retrieval-data','./dataset/test/rsicd_test.csv','--retrieval-images-dir','./images',
 '--csv-separator',',','--csv-img-key','filename',
 '--retrieval-csv-separator',',','--retrieval-csv-img-key','filename',
 '--retrieval-frequency','1','--save-frequency','0',
 '--epochs','1','--batch-size','16','--workers','2','--precision','amp',
 '--lr','2e-7','--warmup','100','--weight_decay','0.5','--max-grad-norm','50',
 '--image-model','ViT-B-32','--image-model-builder','openclip',
 '--text-model','ViT-B-32','--text-model-builder','openclip',
 '--pretrained-image-model','--pretrained-text-model','--load-clip-weights',
 '--load-clip-weights-from','./cache/weights/RET-2.pt',
 '--loss','InfoNCE','--seed','1',
 '--name',NAME,'--logs','/content/drive/MyDrive/prclip_logs']
print('TRAIN COMMAND:', ' '.join(cmd), flush=True)
subprocess.run(cmd,check=True)
print('TRAIN AND EVALUATION COMPLETED')

/content/PR-CLIP
TRAIN COMMAND: /usr/bin/python3 -m training.main --train-data ./dataset/train/ret2.csv --images-dir ./images --retrieval-data ./dataset/test/rsicd_test.csv --retrieval-images-dir ./images --csv-separator , --csv-img-key filename --retrieval-csv-separator , --retrieval-csv-img-key filename --retrieval-frequency 1 --save-frequency 0 --epochs 1 --batch-size 16 --workers 2 --precision amp --lr 2e-7 --warmup 100 --weight_decay 0.5 --max-grad-norm 50 --image-model ViT-B-32 --image-model-builder openclip --text-model ViT-B-32 --text-model-builder openclip --pretrained-image-model --pretrained-text-model --load-clip-weights --load-clip-weights-from ./cache/weights/RET-2.pt --loss InfoNCE --seed 1 --name rsicd_v3_FIXED_AdapterEndToEnd_ep1_bs16_lr2e7 --logs /content/drive/MyDrive/prclip_logs
TRAIN AND EVALUATION COMPLETED


In [12]:
from pathlib import Path
p=Path('/content/drive/MyDrive/prclip_logs')/NAME
assert p.exists(), f'Output missing: {p}'
files=list(p.rglob('*.pt'))
print('Checkpoint files:', [(str(x),round(x.stat().st_size/1e6,1)) for x in files])
assert any(x.name=='best.pt' for x in files), 'best.pt not saved'
print('Saved model under', p)


Checkpoint files: [('/content/drive/MyDrive/prclip_logs/rsicd_v3_FIXED_AdapterEndToEnd_ep1_bs16_lr2e7/checkpoints/best.pt', 430.4)]
Saved model under /content/drive/MyDrive/prclip_logs/rsicd_v3_FIXED_AdapterEndToEnd_ep1_bs16_lr2e7


### Kiểm tra giới hạn
- `best.pt` chứa adapter và trọng số model, nhưng muốn reload phải sử dụng **code v3 có module adapter**, không thể load bằng class PR-CLIP gốc.
- Source v3 dùng reconstruction gốc, cải tiến nằm ở retrieval adapters, không phải reconstruction.
- Notebook này **chưa được chạy trên GPU trong môi trường biên soạn**. Chỉ kiểm tra cú pháp, cấu trúc và unit test offline.
- Tuy cùng thiết lập 1 epoch, training loop AMP được sửa nên chưa có đối chứng tuyệt đối với mR 38.70 trước đó.
